# U-Net Audio Denoiser — Kaggle Training

This notebook trains a U-Net model to denoise audio using mel spectrograms.

### Dataset structure expected
After adding your dataset, it should look like:
```
/kaggle/input/<your-dataset>/
    clean/   *.wav   (speech / clean audio)
    noise/   *.wav   (background noise)
```

### Steps
1. Set `DATASET_PATH` in the **Config** cell to match your dataset path
2. Run all cells top to bottom
3. The best checkpoint will be saved to `/kaggle/working/checkpoints/best.pt`


## Download dataset — LibriSpeech dev-clean (~337 MB)

Uses LibriSpeech for both clean and noise:
- **Clean**: first 75% of speakers → the target signal
- **Noise**: remaining 25% of speakers → babble/background speech noise

This trains the model to suppress background speech, which is a realistic denoising task.
Skip this cell if you uploaded your own dataset.

In [ ]:
import subprocess, shutil
from pathlib import Path

DATASET_PATH = "/kaggle/working/dataset"
CLEAN_DIR    = Path(DATASET_PATH) / "clean"
NOISE_DIR    = Path(DATASET_PATH) / "noise"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
NOISE_DIR.mkdir(parents=True, exist_ok=True)

# ── Download LibriSpeech dev-clean ─────────────────────────────────────────
TAR_PATH = "/kaggle/working/dev-clean.tar.gz"
EXTRACT  = "/kaggle/working/librispeech_raw"

if not Path(TAR_PATH).exists():
    print("Downloading LibriSpeech dev-clean (~337 MB)...")
    subprocess.run([
        "wget", "-q", "--show-progress",
        "https://www.openslr.org/resources/12/dev-clean.tar.gz",
        "-O", TAR_PATH,
    ], check=True)
    print("Download complete.")
else:
    print("Archive already exists, skipping download.")

if not Path(EXTRACT).exists():
    print("Extracting...")
    subprocess.run(["tar", "-xf", TAR_PATH, "-C", "/kaggle/working/"], check=True)
    shutil.move("/kaggle/working/LibriSpeech/dev-clean", EXTRACT)
    shutil.rmtree("/kaggle/working/LibriSpeech", ignore_errors=True)
    print("Extracted.")

# ── Split speakers: 75% clean, 25% noise ──────────────────────────────────
speakers = sorted(Path(EXTRACT).iterdir())
split    = int(len(speakers) * 0.75)
clean_spk, noise_spk = speakers[:split], speakers[split:]

print(f"Total speakers: {len(speakers)}  |  Clean: {len(clean_spk)}  |  Noise: {len(noise_spk)}")

def link_flac(speakers, dest):
    dest = Path(dest)
    count = 0
    for spk in speakers:
        for f in spk.rglob("*.flac"):
            target = dest / f"{spk.name}_{f.parent.name}_{f.name}"
            if not target.exists():
                target.symlink_to(f)
            count += 1
    return count

n_clean = link_flac(clean_spk, CLEAN_DIR)
n_noise = link_flac(noise_spk, NOISE_DIR)

print(f"Clean files: {n_clean:,}")
print(f"Noise files: {n_noise:,}")
print(f"Dataset ready at: {DATASET_PATH}")


In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("soundfile")

import os, torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


## Config — edit this cell before running

In [ ]:
import os, torch

# ── Dataset ────────────────────────────────────────────────────────────────
# If you ran the download cell above, DATASET_PATH is already set.
# If you uploaded your own dataset, uncomment and set this instead:
# DATASET_PATH = "/kaggle/input/your-dataset-name"

# ── Model ──────────────────────────────────────────────────────────────────
BASE_CHANNELS = 32       # 32 → 1.9M params (good balance for Kaggle GPU)
DEPTH         = 3
DROPOUT       = 0.1

# ── Training ───────────────────────────────────────────────────────────────
EPOCHS        = 100
BATCH_SIZE    = 32
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
WORKERS       = 2
CHUNK_FRAMES  = 256

# ── Paths ──────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
LOG_DIR        = "/kaggle/working/runs"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", DEVICE)


## Audio utilities

In [ ]:
import random
import numpy as np
import soundfile as sf
import torch
import torchaudio.transforms as T

SAMPLE_RATE = 16_000
N_FFT       = 1024
HOP_LENGTH  = 256
N_MELS      = 128
F_MIN       = 20.0
F_MAX       = 8_000.0


def make_mel_transform(sr=SAMPLE_RATE):
    return T.MelSpectrogram(
        sample_rate=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX, power=2.0,
    )


def load_audio(path, target_sr=SAMPLE_RATE):
    data, sr = sf.read(str(path), always_2d=True)
    wav = torch.from_numpy(data.T).float()
    if wav.shape[0] > 1:
        wav = wav.mean(0, keepdim=True)
    if sr != target_sr:
        wav = T.Resample(sr, target_sr)(wav)
    return wav


def wav_to_mel(wav, transform=None):
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    if transform is None:
        transform = make_mel_transform()
    mel = transform(wav)
    return torch.log1p(mel)


def normalize_spec(spec):
    min_val = spec.min().item()
    max_val = spec.max().item()
    denom = max_val - min_val
    if denom < 1e-9:
        return spec, min_val, max_val
    return (spec - min_val) / denom, min_val, max_val


def chunk_spectrogram(spec, chunk_frames=256, hop_frames=128):
    _, _, T = spec.shape
    chunks, offsets = [], []
    start = 0
    while start < T:
        chunk = spec[:, :, start:start + chunk_frames]
        if chunk.shape[-1] < chunk_frames:
            chunk = torch.nn.functional.pad(chunk, (0, chunk_frames - chunk.shape[-1]))
        chunks.append(chunk)
        offsets.append(start)
        start += hop_frames
    return chunks, offsets


print("Audio utilities ready.")


## Dataset

In [ ]:
from pathlib import Path
from torch.utils.data import Dataset, DataLoader


class DenoisingDataset(Dataset):
    def __init__(self, root, split="train", chunk_frames=256,
                 snr_min=-5.0, snr_max=20.0, clip_seconds=4.0, sample_rate=SAMPLE_RATE):
        root = Path(root)
        self.chunk_frames  = chunk_frames
        self.snr_min       = snr_min
        self.snr_max       = snr_max
        self.clip_samples  = int(clip_seconds * sample_rate)
        self.sr            = sample_rate
        self.mel_transform = make_mel_transform(sample_rate)

        clean_files = sorted(root.glob("clean/**/*.wav")) or sorted(root.glob("clean/**/*.flac"))
        noise_files = sorted(root.glob("noise/**/*.wav")) or sorted(root.glob("noise/**/*.flac"))

        assert len(clean_files) > 0, f"No clean audio found in {root/'clean'}"
        assert len(noise_files) > 0, f"No noise audio found in {root/'noise'}"

        split_idx = int(len(clean_files) * 0.9)
        self.clean_files = clean_files[:split_idx] if split == "train" else clean_files[split_idx:]
        self.noise_files = noise_files
        self.length = len(self.clean_files) * 10

    def __len__(self):
        return self.length

    def _load_clip(self, path):
        wav = load_audio(path, self.sr)
        T = wav.shape[-1]
        if T < self.clip_samples:
            wav = wav.repeat(1, self.clip_samples // T + 1)
        start = random.randint(0, max(0, wav.shape[-1] - self.clip_samples))
        return wav[:, start:start + self.clip_samples]

    def _mix_at_snr(self, clean, noise, snr_db):
        eps = 1e-9
        p_c = clean.pow(2).mean().clamp(min=eps)
        p_n = noise.pow(2).mean().clamp(min=eps)
        scale = (p_c / (10 ** (snr_db / 10)) / p_n).sqrt()
        return clean + scale * noise

    def __getitem__(self, idx):
        clean_path = self.clean_files[idx % len(self.clean_files)]
        noise_path = random.choice(self.noise_files)

        clean_wav = self._load_clip(clean_path)
        noise_wav = self._load_clip(noise_path)

        snr       = random.uniform(self.snr_min, self.snr_max)
        noisy_wav = self._mix_at_snr(clean_wav, noise_wav, snr).clamp(-1, 1)

        clean_spec = wav_to_mel(clean_wav, self.mel_transform)
        noisy_spec = wav_to_mel(noisy_wav, self.mel_transform)

        noisy_norm, min_val, max_val = normalize_spec(noisy_spec)
        clean_norm, _, _             = normalize_spec(clean_spec)

        _, _, T = noisy_norm.shape
        if T > self.chunk_frames:
            start      = random.randint(0, T - self.chunk_frames)
            noisy_norm = noisy_norm[:, :, start:start + self.chunk_frames]
            clean_norm = clean_norm[:, :, start:start + self.chunk_frames]
        else:
            pad        = self.chunk_frames - T
            noisy_norm = torch.nn.functional.pad(noisy_norm, (0, pad))
            clean_norm = torch.nn.functional.pad(clean_norm, (0, pad))

        return {"noisy": noisy_norm, "clean": clean_norm}


# Verify dataset
train_ds = DenoisingDataset(DATASET_PATH, split="train", chunk_frames=CHUNK_FRAMES)
val_ds   = DenoisingDataset(DATASET_PATH, split="val",   chunk_frames=CHUNK_FRAMES)
print(f"Train: {len(train_ds):,} samples | Val: {len(val_ds):,} samples")
print(f"Sample shape: noisy={train_ds[0]['noisy'].shape}  clean={train_ds[0]['clean'].shape}")


## Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x): return self.block(x)


class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.conv = ConvBlock(in_ch, out_ch, dropout)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        skip = self.conv(x)
        return self.pool(skip), skip


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv = ConvBlock(in_ch, out_ch, dropout)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=True)
        return self.conv(torch.cat([x, skip], dim=1))


class UNet(nn.Module):
    def __init__(self, base_channels=32, depth=3, dropout=0.1):
        super().__init__()
        self.depth = depth
        ch = [base_channels * (2 ** i) for i in range(depth + 1)]

        self.encoders = nn.ModuleList()
        in_ch = 1
        for i in range(depth):
            drop = dropout if i >= depth // 2 else 0.0
            self.encoders.append(DownBlock(in_ch, ch[i], drop))
            in_ch = ch[i]

        self.bottleneck = ConvBlock(ch[depth - 1], ch[depth], dropout)

        self.decoders = nn.ModuleList()
        for i in reversed(range(depth)):
            drop = dropout if i >= depth // 2 else 0.0
            self.decoders.append(UpBlock(ch[i + 1] + ch[i], ch[i], drop))

        self.head = nn.Sequential(nn.Conv2d(ch[0], 1, 1), nn.Sigmoid())

    def forward(self, x):
        skips, out = [], x
        for enc in self.encoders:
            out, skip = enc(out)
            skips.append(skip)
        out = self.bottleneck(out)
        for dec, skip in zip(self.decoders, reversed(skips)):
            out = dec(out, skip)
        return x * self.head(out)


model = UNet(BASE_CHANNELS, DEPTH, DROPOUT).to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f"Model: base_ch={BASE_CHANNELS}, depth={DEPTH} → {total:,} parameters ({total/1e6:.1f}M)")


## Loss & training loop

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total = 0.0
    for batch in loader:
        noisy = batch["noisy"].to(device)
        clean = batch["clean"].to(device)
        pred  = model(noisy)
        loss  = F.l1_loss(pred, clean)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total = 0.0
    for batch in loader:
        noisy = batch["noisy"].to(device)
        clean = batch["clean"].to(device)
        pred  = model(noisy)
        total += F.l1_loss(pred, clean).item()
    return total / len(loader)


def save_checkpoint(state, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(state, path)
    print(f"  Saved: {path}")


print("Training functions ready.")


## Train

In [ ]:
import time

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=WORKERS, pin_memory=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

train_losses, val_losses = [], []
best_val_loss = float("inf")

print(f"Starting training — {EPOCHS} epochs, batch={BATCH_SIZE}, device={DEVICE}\n")

for epoch in range(EPOCHS):
    t0 = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_loss   = validate(model, val_loader, DEVICE)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    elapsed = time.time() - t0
    print(f"Epoch {epoch+1:3d}/{EPOCHS} | train {train_loss:.4f} | val {val_loss:.4f} | {elapsed:.1f}s")

    ckpt = {
        "epoch":      epoch + 1,
        "model":      model.state_dict(),
        "optimizer":  optimizer.state_dict(),
        "val_loss":   val_loss,
        "args": {
            "base_ch": BASE_CHANNELS,
            "depth":   DEPTH,
            "dropout": DROPOUT,
        },
    }
    save_checkpoint(ckpt, f"{CHECKPOINT_DIR}/last.pt")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(ckpt, f"{CHECKPOINT_DIR}/best.pt")
        print(f"  ✓ New best val loss: {val_loss:.4f}")

print(f"\nDone. Best val loss: {best_val_loss:.4f}")
print(f"Checkpoint saved to: {CHECKPOINT_DIR}/best.pt")


## Loss curves

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(train_losses) + 1)
plt.figure(figsize=(10, 4))
plt.plot(epochs_range, train_losses, label="Train L1")
plt.plot(epochs_range, val_losses,   label="Val L1")
plt.xlabel("Epoch")
plt.ylabel("L1 Loss")
plt.title("Training curves")
plt.legend()
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/loss_curves.png", dpi=120)
plt.show()
print("Saved loss_curves.png")


## Download checkpoint

After training, go to **Output** tab on the right panel and download `checkpoints/best.pt`. Then drop it into your local `checkpoints/` folder to use with `denoise.py` and `app.py`.

In [ ]:
import shutil

# Zip checkpoint + loss curve for easy download
shutil.make_archive("/kaggle/working/model_output", "zip", CHECKPOINT_DIR)
print("Download /kaggle/working/model_output.zip from the Output tab.")
print(f"Files in checkpoint dir: {os.listdir(CHECKPOINT_DIR)}")
